# Quest Notebook Solution

In [1]:
# Import neccessary modules, add to this cell as needed
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

## Part 1: Load the Sample Dataset

In [2]:
# Initiate a new Spark session and set the case sensitivity option
spark = (
    SparkSession.builder
        .appName("cyberquest")
        .getOrCreate()
)
spark.conf.set("spark.sql.caseSensitive", True)

In [3]:
df_bronze = spark.read.json("./data/sysmon_spearphish_cribl.json")

In [4]:
raw_schema = "Image string, UserID string, QueryName string, QueryStatus decimal, QueryResults string, SystemTime string, ProcessId string, Channel string"

df_silver = (df_bronze
    .select(
        F.to_timestamp(F.col("_time")).alias("_time"),
        "Computer",
        "EventCode",
        "User",
        F.from_json(F.col("_raw"), raw_schema).alias("_parsed"),
        "_raw",
    )
    .select("*", "_parsed.*")
    .drop("_parsed")
)
df_silver.createOrReplaceTempView("sysmon_silver")

In [ ]:
# PySpark Example
df_silver.filter("EventCode == '22'").limit(5).show()

+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|               _time|            Computer|EventCode|User|                _raw|               Image|  UserID|           QueryName|QueryStatus|        QueryResults|          SystemTime|ProcessId|             Channel|
+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Temp\OfficeSet...|S-1-5-18|      ecs.office.com|          0|type:  5 ecs.offi...|'2023-01-27T11:22...|     6048|Microsoft-Windows...|
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Windows\System...|S-1-5-18|f.c2r.ts.cdn.offi...|      

In [74]:
# SQL Example
spark.sql("""
SELECT *
FROM sysmon_silver
WHERE EventCode = 22
LIMIT 100
""").show()

+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|               _time|            Computer|EventCode|User|                _raw|               Image|  UserID|           QueryName|QueryStatus|        QueryResults|          SystemTime|ProcessId|             Channel|
+--------------------+--------------------+---------+----+--------------------+--------------------+--------+--------------------+-----------+--------------------+--------------------+---------+--------------------+
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Temp\OfficeSet...|S-1-5-18|      ecs.office.com|          0|type:  5 ecs.offi...|'2023-01-27T11:22...|     6048|Microsoft-Windows...|
|2023-01-27 22:22:...|win-host-ctus-att...|       22|NULL|{"Name":"'Microso...|C:\Windows\System...|S-1-5-18|f.c2r.ts.cdn.offi...|      

## Part 2: Detection Engineering

In [ ]:
# prove hypothesis
spark.sql("""
SELECT *
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE '%microsoft office%'
AND LOWER(Image) LIKE '%.exe'
AND NOT(LOWER(QueryName) LIKE '%.office.net' OR LOWER(QueryName) LIKE '%.office.com')
""").show(truncate=False)


+-----------------------+------------------------------+---------+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------+--------+-----------------+-----------+----------------------------------------+--------------------------------+---------+------------------------------------+
|_time                  |Computer             

In [69]:
# create new dataframe with result
df_detect = spark.sql("""
SELECT _time, Computer, EventCode, User, Image, UserID, QueryName, QueryResults
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE '%microsoft office%'
AND LOWER(Image) LIKE '%.exe'
AND NOT(LOWER(QueryName) LIKE '%.office.net' OR LOWER(QueryName) LIKE '%.office.com')
""")

## Part 3: Additional Steps

### Part 3.1: Normalization

In [61]:
# create dataframe with DNS queries - normalised to Splunk CIM Network Resolution (DNS) data model
spark.sql("""
SELECT _time, Computer AS src, QueryName AS query, QueryResults AS answer, 'dns' AS tag
FROM sysmon_silver
WHERE EventCode = 22
""").show(truncate=False)

+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|_time                  |src                           |query                            |answer                                                                                                                                                                    |tag|
+-----------------------+------------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|2023-01-27 22:22:35.226|win-host-ctus-attack-range-212|ecs.office.com                   |type:  5 ecs.office.trafficmanager.net;type:  5 s-0005-office.config.skype.com;type:  5 ecs-office.s-0005.s-msed

### Part 3.2: Alert Table

In [64]:
# create new dataframe as alert table
df_alert_table = spark.sql("""
SELECT _time, 'Sysmon' AS LogSource, 'Potentially malicious webcall intiated by Microsoft Office application' AS Title,'High' AS Severity, Computer AS `Host`, EventCode AS `SysmonEventCode`, Image AS `InitiatingProcessPath`, UserID, `QueryName` AS DNSQueryDomain, QueryResults AS `DNSQueryResults`, 'Execution' AS MitreTactic, 'T1203' AS MitreID, 'Exploitation for Client Execution' AS MitreTechnique
FROM sysmon_silver
WHERE EventCode = 22
AND LOWER(Image) LIKE '%microsoft office%'
AND LOWER(Image) LIKE '%.exe'
AND NOT(LOWER(QueryName) LIKE '%.office.net' OR LOWER(QueryName) LIKE '%.office.com')
""")

df_alert_table.show(truncate=False)

+-----------------------+---------+----------------------------------------------------------------------+--------+------------------------------+---------------+-----------------------------------------------------------+--------+-----------------+----------------------------------------+-----------+-------+---------------------------------+
|_time                  |LogSource|Title                                                                 |Severity|Host                          |SysmonEventCode|InitiatingProcessPath                                      |UserID  |DNSQueryDomain   |DNSQueryResults                         |MitreTactic|MitreID|MitreTechnique                   |
+-----------------------+---------+----------------------------------------------------------------------+--------+------------------------------+---------------+-----------------------------------------------------------+--------+-----------------+----------------------------------------+-----------+-------+

### Part 3.3: Enrichment

In [ ]:
# Not attempted

## Summary

See readme.md for summary.

### Part 1: Load the Sample Dataset (provided)

The initial dataframe provided (df_silver) contains Sysmon logs.

### Part 2: Detection Engineering

df_silver dataframe shows sysmon logs from a Windows device. In order to prove the hypothesis, the dataframe was filtered for DNS query events. As threat actors utilising Microsoft Office applications are to be targeted by the detection, the dataframe is filtered for "Microsoft Office" string anywhere in Image (the path of the executable that triggered the event), ending with ".exe". MS Office applications often query Microsoft, shown in legitimate query events in the logs, therefore, the next filter filters out legitmate DNS queries ending in "office.net" or "office.com". 

df_detect was created with the results, including _time, Computer, EventCode, User, Image, UserID, QueryName and QueryResults columns, relevant for a security analyst.

df_detect shows a single event showing DNS query initiated by WINWORD.EXE to www.mediafire.com, a filehosting site. This potentially indicates a macro running in MS Word, and pulling down a malicious file hosted on Mediafire. IPv4 addresses route to Cloudflare, indicating that traffic to Mediafire is first routed to/proxied by Cloudflare.

### Part 3: Additional Steps

#### 3.1 Data Normalization
Splunk CIM Network Resolution (DNS) data model was utilised for normalisation of the output. 

#### 3.2 Write the result to a fictitious `alert` table
 - Package the result of the detection as an alert row in a new dataframe that would theoretically be used by an analyst during triage
    - Think about what an analyst would need, what metadata would be useful at a high level, how this might be presented on a dashboard

Information necessary for rapid triage and investigation was added as part of the output to the alert table (df_alert_table) including log source, alert title severity. MITRE ATT&CK Tactic and Technique were added for TTP mapping of alerts. Other columns were renamed for clarity.

#### 3.3 Threat Intel Enrichment on Domain in Query
  - Given that the source data includes domain names, enrich the detection or alert with information from any threat intelligence source

I was unable to perform enrichment directly in this environment. In a real world scenario, an alternative would be to perform a JOIN on a threat intel IOC table, if such a table available in the SIEM. 